# Geometric Solid Modeling and Parameterized Functions

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

In 3D computer vision, physics simulations, robotics, and spatial computer graphics, computing volumetric properties of bounding shapes (cuboids, bounding spheres, cylindrical hulls) is a frequent requirement.

This notebook builds a suite of four parameterized volumetric functions, examining mathematical definitions, numerical constant precision, parameter validation, and object-oriented encapsulation.

### Objectives
1. Implement volume calculations for cubes, rectangular prisms, cylinders, and spheres.
2. Analyze precision differences between approximating $\pi \approx 3.14$ versus `math.pi`.
3. Add robust defensive validation against non-positive dimensions.
4. Encapsulate geometric models within an object-oriented `SolidGeometry` class.


## 1. Mathematical Formulations

Given 3D Euclidean space:

1. **Cube:** Side length $s$. All three spatial dimensions are equal ($s = l = w = h$):
   $$V_{\text{cube}}(s) = s^3$$

2. **Rectangular Prism (Cuboid):** Length $l$, width $w$, height $h$:
   $$V_{\text{prism}}(l, w, h) = l \cdot w \cdot h$$

3. **Right Circular Cylinder:** Circular base radius $r$, axial height $h$:
   $$V_{\text{cylinder}}(r, h) = \pi r^2 h$$

4. **Sphere:** Radial distance $r$ from centroid to surface:
   $$V_{\text{sphere}}(r) = \frac{4}{3} \pi r^3$$


In [ ]:
import math


def calc_cube_volume(s: float) -> float:
    """Calculates the volume of a cube with side length s."""
    if s <= 0:
        raise ValueError(f"Side length must be positive, got {s}")
    return float(s**3)


def calc_rectangular_prism_volume(length: float, width: float, height: float) -> float:
    """Calculates the volume of a rectangular prism with length l, width w, and height h."""
    if length <= 0 or width <= 0 or height <= 0:
        raise ValueError(
            f"Dimensions must be positive, got length={length}, width={width}, height={height}"
        )
    return float(length * width * height)


def calc_cylinder_volume(r: float, h: float, pi: float = math.pi) -> float:
    """Calculates the volume of a circular cylinder with base radius r and height h."""
    if r <= 0 or h <= 0:
        raise ValueError(f"Radius and height must be positive, got r={r}, h={h}")
    return float(pi * (r**2) * h)


def calc_sphere_volume(r: float, pi: float = math.pi) -> float:
    """Calculates the volume of a sphere with radius r."""
    if r <= 0:
        raise ValueError(f"Radius must be positive, got {r}")
    return float((4.0 / 3.0) * pi * (r**3))

## 2. Numerical Precision: `math.pi` vs Truncated Approximations

Software implementations often vary based on constant definitions:
- Truncated constant: $\pi \approx 3.14$
- Standard library IEEE 754 double precision: `math.pi` $= 3.141592653589793$

Let us quantify the absolute and relative error introduced by using $3.14$ across cylinder and sphere calculations.


In [ ]:
r_test = 5.0
h_test = 12.0

# Cylinder comparison
v_cyl_exact = calc_cylinder_volume(r_test, h_test, pi=math.pi)
v_cyl_approx = calc_cylinder_volume(r_test, h_test, pi=3.14)
cyl_rel_error = abs(v_cyl_exact - v_cyl_approx) / v_cyl_exact

# Sphere comparison
v_sph_exact = calc_sphere_volume(r_test, pi=math.pi)
v_sph_approx = calc_sphere_volume(r_test, pi=3.14)
sph_rel_error = abs(v_sph_exact - v_sph_approx) / v_sph_exact

print(f"Cylinder (Exact):   {v_cyl_exact:.6f}")
print(f"Cylinder (Approx):  {v_cyl_approx:.6f} | Relative Error: {cyl_rel_error:.4%}")
print("-" * 55)
print(f"Sphere (Exact):     {v_sph_exact:.6f}")
print(f"Sphere (Approx):    {v_sph_approx:.6f} | Relative Error: {sph_rel_error:.4%}")

## 3. Object-Oriented Solid Geometry Representation

In production code, grouping geometric shape parameters with their geometric equations into domain classes improves maintainability and reusability.


In [ ]:
from dataclasses import dataclass
from typing import ClassVar


@dataclass(frozen=True)
class Cube:
    side: float

    def volume(self) -> float:
        return calc_cube_volume(self.side)

    def surface_area(self) -> float:
        return 6.0 * (self.side**2)


@dataclass(frozen=True)
class Sphere:
    radius: float
    PI: ClassVar[float] = math.pi

    def volume(self) -> float:
        return calc_sphere_volume(self.radius, pi=self.PI)

    def surface_area(self) -> float:
        return 4.0 * self.PI * (self.radius**2)


# Instantiate models
c = Cube(side=4.0)
s = Sphere(radius=3.0)

print(f"Cube volume:        {c.volume():.2f}, surface area: {c.surface_area():.2f}")
print(f"Sphere volume:      {s.volume():.2f}, surface area: {s.surface_area():.2f}")

## 4. Verification and Assertions

We execute automated test assertions to verify mathematical correctness across standard shapes.


In [ ]:
# Verification assertions
assert math.isclose(calc_cube_volume(3.0), 27.0)
assert math.isclose(calc_rectangular_prism_volume(2.0, 3.0, 4.0), 24.0)
assert math.isclose(calc_cylinder_volume(2.0, 5.0, pi=3.14), 62.8, rel_tol=1e-5)
assert math.isclose(calc_sphere_volume(3.0, pi=3.14), 113.04, rel_tol=1e-5)

print("All geometric volume assertions passed successfully!")